# Cohere API and Hugging Face API models, evaluated with AuditKIT

Both providers serve an **OpenAI-compatible** chat endpoint, so AuditKIT evaluates them through the same `api:` backend (`APIModel`):

| Provider | Base URL | Key |
|---|---|---|
| Cohere | `https://api.cohere.ai/compatibility/v1` | `CO_API_KEY` |
| Hugging Face Inference Providers | `https://router.huggingface.co/v1` | `HF_TOKEN` (with *Make calls to Inference Providers* permission) |

| § | What it tests | Metrics |
|---|---|---|
| 0 | Which configured models answer at all (a 1-request probe) | – |
| 1 | Factual and **multilingual** QA, side by side, with latency | `quasi_exact_match`, `f1_score` (case/punctuation-insensitive), latency |
| 2 | Structured output | `is_json` |
| 3 | **Native tool calling**: single, parallel, choosing the right tool, and not calling at all | `tool_call_f1`, `trajectory_match`, `parallel_tool_calls`, `tool_call_validity`, `tool_permission` |
| 4 | Summary table across all models | – |

**Runtime:** CPU is enough; nothing runs locally.

**Secrets (Colab → 🔑):** `GITHUB_TOKEN` (to install AuditKIT), plus `CO_API_KEY` and/or `HF_TOKEN`. A provider without its key is skipped. A Cohere **trial** key is rate-limited (about 20 chat calls a minute), so the runs use one request at a time and retry on 429.

In [ ]:
# ---- settings: edit the model lists freely; unavailable models are skipped by the probe ----
import os
COHERE_BASE = os.environ.get("NB_COHERE_BASE", "https://api.cohere.ai/compatibility/v1")
HF_BASE = os.environ.get("NB_HF_BASE", "https://router.huggingface.co/v1")
COHERE_MODELS = ["command-a-03-2025", "command-r7b-12-2024", "c4ai-aya-expanse-32b"]
HF_MODELS = ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "CohereLabs/aya-expanse-32b"]
MAX_RETRIES, RETRY_DELAY = 6, 10     # 429 / 5xx are retried per request

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at BRANCH (#1's branch, where #10 and every #15 fix are merged) and installs it.
# Locally: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, subprocess, sys
BRANCH = "feat/rag-agent-evals"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'HF_TOKEN', 'CO_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass                          # secret not set / not granted to this notebook
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[requests]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")
import auditkit as ak
print("auditkit", ak.__version__, "| secrets:", {k: bool(os.environ.get(k)) for k in ('HF_TOKEN', 'CO_API_KEY')})

In [ ]:
import json
import pandas as pd
from auditkit import Sample
from auditkit.runspec import RunConfig
from auditkit.model import Request
from auditkit.model.api_gen import APIModel
pd.set_option("display.max_colwidth", 80)

def cfg(**kw):
    """One request at a time (trial-key friendly), retries on 429/5xx, deterministic sampling."""
    return RunConfig(temperature=0.0, concurrency=1, max_retries=MAX_RETRIES, retry_delay=RETRY_DELAY, **kw)

def make(provider, model):
    base, key = (COHERE_BASE, os.environ.get("CO_API_KEY")) if provider == "cohere" else (HF_BASE, os.environ.get("HF_TOKEN"))
    m = APIModel(model, api_base=base, api_key=key, name=f"{provider}:{model}")
    m.max_retries, m.retry_delay = MAX_RETRIES, RETRY_DELAY
    return m

## 0 · Probe the configured models

One short request per model. A model that errors (wrong name, no access, not served by any provider for your account) is listed with the reason and left out of the rest of the notebook.

In [ ]:
MODELS, SKIPPED = {}, {}
for provider, names, key in (("cohere", COHERE_MODELS, "CO_API_KEY"), ("hf", HF_MODELS, "HF_TOKEN")):
    for n in names:
        if not os.environ.get(key):
            SKIPPED[f"{provider}:{n}"] = f"{key} not set"
            continue
        m = make(provider, n)
        g = m.generate([Request(prompt="Reply with the single word OK.", params={"max_tokens": 8, "temperature": 0.0})])[0].completions[0]
        if g.finish_reason == "error" or g.error:
            SKIPPED[m.name] = (g.error or "error")[:200]
        else:
            MODELS[m.name] = m
print("available:", list(MODELS))
if SKIPPED:
    display(pd.DataFrame([{"model": k, "why skipped": v} for k, v in SKIPPED.items()]))
assert MODELS, "no model answered: set CO_API_KEY and/or HF_TOKEN, and check the model names above"

## 1 · Factual and multilingual QA

`compare_models` runs every model on the same samples and lines the results up. Answers are short, so:
- `quasi_exact_match` normalises case, punctuation and articles, then compares the whole answer;
- `f1_score` is token overlap, SQuAD-normalised by default (so "Paris." matches "paris").

The multilingual rows ask in French, Spanish, Hindi, Japanese, Arabic and German and expect the answer in that language, which is where Aya models should stand out.

In [ ]:
ASK = "Answer with only the answer, no explanation. "
QA = [
    ("What is the capital of Australia?", "Canberra"),
    ("How many sides does a hexagon have? Answer with a number.", "6"),
    ("What is 17 multiplied by 3? Answer with a number.", "51"),
    ("Which planet is known as the Red Planet?", "Mars"),
    ("Who wrote 'Pride and Prejudice'?", "Jane Austen"),
    ("What is the chemical symbol for gold?", "Au"),
    ("Quelle est la capitale du Canada ? Réponds en un mot.", "Ottawa"),
    ("¿Cuál es el río más largo de Sudamérica? Responde solo con el nombre.", "Amazonas"),
    ("भारत की राजधानी क्या है? केवल नाम लिखें।", "नई दिल्ली"),
    ("日本の首都はどこですか？都市名だけで答えてください。", "東京"),
    ("ما هي عاصمة مصر؟ أجب بكلمة واحدة.", "القاهرة"),
    ("Wie viele Tage hat eine Woche? Antworte nur mit der Zahl.", "7"),
]
qa = [Sample(id=f"qa{i:02d}", input=ASK + q, target=a) for i, (q, a) in enumerate(QA)]
cmp_qa = ak.compare_models(list(MODELS.values()), qa, scorers=["quasi_exact_match", "f1_score"],
                           model_names=list(MODELS), config=cfg(max_tokens=48))
display(pd.DataFrame(cmp_qa.comparison_table()))
display(pd.DataFrame(cmp_qa.performance_table()))
if cmp_qa.errors:
    print("models that failed entirely:", cmp_qa.errors)

In [ ]:
# every answer, per model: where the misses are
rows = []
for name, run in cmp_qa.runs.items():
    for p in run.predictions:
        rows.append({"sample": p.sample_id, "model": name, "expected": p.expected, "answer": (p.raw_output or "").strip()[:60]})
display(pd.DataFrame(rows).pivot(index=["sample", "expected"], columns="model", values="answer"))

## 2 · Structured output

`is_json` checks that the whole reply parses as JSON, which is what an app parsing the reply needs. Models that wrap JSON in a markdown fence or add a sentence fail it.

In [ ]:
JSON_TASKS = [
    "Return a JSON object with keys \"city\" and \"country\" for the Eiffel Tower's location. Output JSON only.",
    "Return a JSON array of the first three prime numbers. Output JSON only.",
    "Return a JSON object {\"sentiment\": ...} where sentiment is positive or negative for: 'The service was wonderful.' Output JSON only.",
]
js = [Sample(id=f"json{i}", input=t) for i, t in enumerate(JSON_TASKS)]
cmp_json = ak.compare_models(list(MODELS.values()), js, scorers=["is_json"], model_names=list(MODELS), config=cfg(max_tokens=96))
display(pd.DataFrame(cmp_json.comparison_table()))

## 3 · Native tool calling

`ToolCallAdapter()` sends the tool schemas in the request's `tools` field. The provider returns `message.tool_calls`, and AuditKIT scores them as one **turn**; several calls in one reply are a parallel call.

`expected_tool_calls` is a list of turns: `[[a, b]]` means *a* and *b* belong in the same reply, and `[]` means the right move is to answer without a tool.

In [ ]:
TOOLS = [
    {"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount between currencies",
     "parameters": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"},
                    "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}}},
    {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
]
W = lambda c: {"name": "get_weather", "arguments": {"city": c}}
TOOL_TASKS = [
    Sample(id="single", input="What's the weather in Lisbon right now?", tools=TOOLS, expected_tool_calls=[[W("Lisbon")]]),
    Sample(id="parallel", input="What's the weather in Oslo and in Cairo? Check both.", tools=TOOLS,
           expected_tool_calls=[[W("Oslo"), W("Cairo")]]),
    Sample(id="pick_tool", input="What time is it in Seoul?", tools=TOOLS,
           expected_tool_calls=[[{"name": "get_time", "arguments": {"city": "Seoul"}}]]),
    Sample(id="args", input="Convert 250 US dollars to euros.", tools=TOOLS,
           expected_tool_calls=[[{"name": "convert_currency", "arguments": {"amount": 250, "from_currency": "USD", "to_currency": "EUR"}}]]),
    Sample(id="no_tool", input="What is the capital of Italy? Answer directly.", tools=TOOLS, expected_tool_calls=[]),
]
TOOL_SCORERS = lambda: [ak.ToolCallF1(), ak.TrajectoryMatch(), ak.ParallelToolCalls(), ak.ToolCallValidity(), ak.ToolPermission()]
cmp_tools = ak.compare_models(list(MODELS.values()), TOOL_TASKS, scorers=TOOL_SCORERS(), model_names=list(MODELS),
                              adapter=ak.ToolCallAdapter(), config=cfg(max_tokens=256))
display(pd.DataFrame(cmp_tools.comparison_table()))

In [ ]:
# the calls each model actually made (from the structured trace)
rows = []
for name, run in cmp_tools.runs.items():
    for p in run.predictions:
        turns = ((p.context or {}).get("trace") or {}).get("tool_calls") or []
        made = [[(c.get("function") or c).get("name") + "(" + str((c.get("function") or c).get("arguments")) + ")" for c in t] for t in turns]
        rows.append({"sample": p.sample_id, "model": name, "calls": made or "-", "text": (p.raw_output or "")[:40]})
display(pd.DataFrame(rows).pivot(index="sample", columns="model", values="calls"))
errs = {n: r.errors for n, r in cmp_tools.runs.items() if r.errors}
if errs:
    print("per-sample errors (e.g. a model/provider without tool support):", json.dumps(errs, indent=1, default=str)[:3000])

## 4 · Summary

In [ ]:
summary = {}
for label, cmp in (("qa", cmp_qa), ("json", cmp_json), ("tools", cmp_tools)):
    for name, run in cmp.runs.items():
        for k, v in run.headline.items():
            summary.setdefault(name, {})[f"{label}.{k}"] = round(v, 3)
        if run.perf and run.perf.get("latency_ms"):
            summary[name][f"{label}.latency_p50_ms"] = run.perf["latency_ms"].get("p50")
keep = ["qa.quasi_exact_match", "qa.f1_score", "qa.latency_p50_ms", "json.is_json", "tools.tool_call_f1",
        "tools.tool_call_exact", "tools.trajectory_strict", "tools.parallel_detection", "tools.tool_call_validity", "tools.tool_permission"]
df = pd.DataFrame(summary).T
display(df[[c for c in keep if c in df.columns]])

### Reading the numbers
- **`quasi_exact_match`** is strict (the whole answer must match); **`f1_score`** gives partial credit.
- **`tool_call_f1`**: right tools with the right arguments, as a set. **`tool_call_exact`**: every expected call and nothing else.
- **`trajectory_strict`**: the calls in the right turns. **`parallel_detection`**: batched exactly when the reference batches.
- **`tool_call_validity`**: calls conform to the schemas. **`tool_permission`**: only offered tools were called.
- The `no_tool` row is the irrelevance check: calling a tool there scores 0 on every tool metric.
- A provider that doesn't support tools for a model shows up under *per-sample errors*, never as a silent 0.